# Description

I want to optimize the batch size and simultaneous number of images that I am feeding to cellpose for given hardware.

As a note: I found that I am saturating the GPU capacity before I saturate memory for a single cellpose workload. Therefore, I am not going to try to batch multiple cellpose processes at the same time, and will make snakemake allocate GPU capacity to a single cellpose task at a time.

In [ ]:
import pandas as pd
import numpy as np
from cellpose import models
model = models.Cellpose(model_type='cyto3', gpu=True)

In [13]:
from matplotlib.pyplot import imread
import time

In [ ]:
cellpose_diam = 41
inital_batch_size = 8
inital_list_size = 1
manifest = '/Users/owenburroughs/Desktop/Personal_Skaar_Lab/EPS007_Code/Test_Output/AVI0G830_TestPlate/manifest.csv'

manifest_df = pd.read_csv(manifest)
cellpose_images = manifest_df['CellMask']

#Set the initial parameters for the first pass
solution_found = False
batch_size = inital_batch_size
list_size = inital_list_size
best_time = 1000000000


#Function to run cellpose on an image set and return the runtime
def time_cellpose(image_array, batch_size):
    start_time = time.time()
    masks, flows, styles, diams = model.eval(
        image_array, 
        channels=[0,0], 
        diameter=cellpose_diam,
        resample=False, 
        batch_size= batch_size,
        min_size = 100
    )
    return time.time() - start_time


runtimes = [["seconds", "list_size", "batch_size"]]


#iterate until we have a solution
while not solution_found:
#Create the array of images for cellpose
    if list_size > len(cellpose_images):
        list_size = len(cellpose_images)
        print("Tried to use a list size greater than the length of the manifest. Use a manifest with more rows for a more complete search.")
    
    image_path_list = list(cellpose_images[0:list_size])
    image_list = []
    for path in image_path_list:
        image_list.append(imread(path))
    image_array = np.stack(image_list)
    
#Evaluate the marginal values for batch size:
    batch_time = time_cellpose(image_list, batch_size) / list_size
    runtimes.append([batch_time, list_size, batch_size])

    #If batch size = 1, then batch for n-1 won't evaluate, so set it to batch_time instead
    if batch_size != 1:
        batch_minus_one_time = time_cellpose(image_list, batch_size - 1) / list_size
    else:
        batch_minus_one_time = batch_time
    runtimes.append([batch_minus_one_time, list_size, batch_size - 1])
    
    batch_plus_one_time = time_cellpose(image_list, batch_size + 1) / list_size
    runtimes.append([batch_plus_one_time, list_size, batch_size + 1])
    
#Now, pick the newst batch size based on these results
    if batch_minus_one_time <= batch_time and batch_plus_one_time >= batch_time:
        #slope is positive, pick a larger batch size
        
    
    
    solution_found = True

runtimes_df = pd.DataFrame(runtimes)
display(runtimes_df)


,0,1,2
0,seconds,list_size,batch_size
1,1.847089,1,8
2,1.830638,1,7
3,1.851371,1,9


## I don't want to write a solver!!!
That might be a fun task for another day. In the meantime, I will just use Scipy

In [ ]:
import cellpose

cellpose_diam = 41
initial_batch_size = 8
initial_list_size = 1
manifest = '/Users/owenburroughs/Desktop/Personal_Skaar_Lab/EPS007_Code/Test_Output/AVI0G830_TestPlate/manifest.csv'

manifest_df = pd.read_csv(manifest)
cellpose_images = manifest_df['CellMask']


#This is the function we are optimizing. This take list_size and batch_size, and returns time_per_image (float)
def time_cellpose(arguments):
    list_size = int(arguments[0])
    batch_size = int(arguments[1])
    
    print(f'List size: {list_size}, Batch size: {batch_size}')
    
    #Create a numpy array of list_size size
    image_path_list = list(cellpose_images[0:list_size])
    image_list = []
    for path in image_path_list:
        image_list.append(imread(path))
    image_array = np.stack(image_list, axis=0)
    
    #First, pre-initialize the cellpose model by running to help keep the runtimes accurate.
    masks, flows, styles, diams = model.eval(
            image_array, 
            channels=[0,0], 
            diameter=cellpose_diam,
            resample=False, 
            batch_size= batch_size,
            min_size = 100
        )
    
    #Now, run the same cellpose process again but time it this time
    start_time = time.time()
    masks, flows, styles, diams = model.eval(
        image_array, 
        channels=[0,0], 
        diameter=cellpose_diam,
        resample=False, 
        batch_size= batch_size,
        min_size = 100,
        do_3D=False
    )
    runtime = (time.time() - start_time)/list_size
    print(f'Runtime was {runtime}')
    
    print(image_array.shape)
    print(image_array.dtype)
    
    return runtime

In [34]:
from scipy.optimize import differential_evolution

max_batch_size = 16
max_list_size = 8

#max_list_size = len(cellpose_images)

bounds = [(1, max_list_size), (1, max_batch_size)]

minimized = differential_evolution(
    time_cellpose,
    bounds,
    disp = True   
)

z_axis not specified, assuming it is dim 0
if this is actually the channel_axis, use 'model.eval(channel_axis=0, ...)'


List size: 7, Batch size: 15


3D stack used, but stitch_threshold=0 and do_3D=False, so masks are made per plane only
z_axis not specified, assuming it is dim 0
if this is actually the channel_axis, use 'model.eval(channel_axis=0, ...)'
3D stack used, but stitch_threshold=0 and do_3D=False, so masks are made per plane only


Runtime was 2.0076158387320384


AttributeError: module 'cellpose' has no attribute '__version__'

In [35]:
print(image_array.shape)
print(image_array.dtype)

(1, 2048, 2048)
uint16
